In [ ]:
import os, json, random, math, time
from dataclasses import dataclass, asdict
from itertools import product

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, balanced_accuracy_score
)


In [ ]:

DATA_DIR = "/content/drive/MyDrive/brain_tumor_data_set"
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 20
BASE_LR = 1e-3

NUM_WORKERS = 2
PIN_MEM = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

ACTIVATIONS = ["relu", "leakyrelu", "sigmoid"]
OPTIMIZERS = ["adam", "sgd", "rmsprop"]
CONV_LAYERS = [1, 2, 3]
FC_LAYERS   = [1, 2, 3]


SEEDS = [0, 1, 2]


TEST_SIZE = 0.25

# early stopping
PATIENCE = 3
MIN_DELTA = 1e-4


DEVICE: cuda


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

full_ds = datasets.ImageFolder(DATA_DIR, transform=train_tfms)
class_names = full_ds.classes
print("Classes:", class_names)

y_all = np.array([full_ds.samples[i][1] for i in range(len(full_ds))])
print("Total:", len(full_ds), "Class counts:", np.bincount(y_all))


Classes: ['no', 'yes']
Total: 253 Class counts: [ 98 155]


In [ ]:
def make_split_loaders(seed: int):

    sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
    idx = np.arange(len(full_ds))
    train_idx, val_idx = next(sss.split(idx, y_all))


    train_ds = datasets.ImageFolder(DATA_DIR, transform=train_tfms)
    val_ds   = datasets.ImageFolder(DATA_DIR, transform=val_tfms)

    train_subset = Subset(train_ds, train_idx)
    val_subset   = Subset(val_ds, val_idx)

    train_loader = DataLoader(
        train_subset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEM
    )
    val_loader = DataLoader(
        val_subset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEM
    )
    return train_loader, val_loader, train_idx, val_idx


In [ ]:
def get_activation(name: str):
    name = name.lower()
    if name == "relu":
        return nn.ReLU(inplace=True)
    if name == "leakyrelu":
        return nn.LeakyReLU(0.1, inplace=True)
    if name == "sigmoid":

        return nn.Sigmoid()
    raise ValueError(f"Unknown activation: {name}")

class SimpleCNN(nn.Module):
    def __init__(self, conv_layers: int, fc_layers: int, activation: str, num_classes: int = 2):
        super().__init__()
        act = get_activation(activation)


        chs = [16, 32, 64]

        layers = []
        in_ch = 3
        for i in range(conv_layers):
            out_ch = chs[i]
            layers += [
                nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1),
                nn.MaxPool2d(kernel_size=2),
                act,
            ]
            in_ch = out_ch

        self.conv = nn.Sequential(*layers)


        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
            feat = self.conv(dummy)
            flat_dim = feat.view(1, -1).shape[1]

        fc = []
        hidden = 128
        if fc_layers == 1:
            fc += [nn.Linear(flat_dim, num_classes)]
        else:
            fc += [nn.Linear(flat_dim, hidden), act]
            for _ in range(fc_layers - 2):
                fc += [nn.Linear(hidden, hidden), act]
            fc += [nn.Linear(hidden, num_classes)]

        self.fc = nn.Sequential(*fc)

    def forward(self, x):
        x = self.conv(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x  # logits


In [ ]:
def make_optimizer(name: str, params, lr: float):
    name = name.lower()
    if name == "adam":
        return optim.Adam(params, lr=lr)
    if name == "sgd":
        return optim.SGD(params, lr=lr, momentum=0.9)
    if name == "rmsprop":
        return optim.RMSprop(params, lr=lr, momentum=0.9)
    raise ValueError(f"Unknown optimizer: {name}")

def logits_to_probs(logits: torch.Tensor):

    probs = torch.softmax(logits, dim=1)[:, 1]
    return probs

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    losses = []
    all_y = []
    all_prob = []
    all_pred = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = criterion(logits, y)
        losses.append(loss.item())

        prob = logits_to_probs(logits).detach().cpu().numpy()
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy()

        all_prob.append(prob)
        all_pred.append(pred)
        all_y.append(y.detach().cpu().numpy())

    all_y = np.concatenate(all_y)
    all_prob = np.concatenate(all_prob)
    all_pred = np.concatenate(all_pred)

    acc = accuracy_score(all_y, all_pred)
    bal_acc = balanced_accuracy_score(all_y, all_pred)
    f1 = f1_score(all_y, all_pred)

    try:
        auc = roc_auc_score(all_y, all_prob)
    except ValueError:
        auc = np.nan

    return {
        "loss": float(np.mean(losses)),
        "acc": float(acc),
        "bal_acc": float(bal_acc),
        "f1": float(f1),
        "auc": float(auc),
    }

def train_one_run(
    conv_layers: int, fc_layers: int, optimizer_name: str, activation: str,
    seed: int, run_dir: str
):
    set_seed(seed)
    train_loader, val_loader, train_idx, val_idx = make_split_loaders(seed)

    model = SimpleCNN(conv_layers, fc_layers, activation, num_classes=2).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    opt = make_optimizer(optimizer_name, model.parameters(), lr=BASE_LR)

    best_val_loss = float("inf")
    best_state = None
    best_epoch = -1
    no_improve = 0

    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            opt.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            opt.step()

            train_losses.append(loss.item())

        train_metrics = evaluate(model, train_loader, criterion)
        val_metrics = evaluate(model, val_loader, criterion)

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "train_acc": train_metrics["acc"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_bal_acc": val_metrics["bal_acc"],
            "val_f1": val_metrics["f1"],
            "val_auc": val_metrics["auc"],
        }
        history.append(row)


        if val_metrics["loss"] < best_val_loss - MIN_DELTA:
            best_val_loss = val_metrics["loss"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    os.makedirs(run_dir, exist_ok=True)

    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)


    if best_state is not None:
        ckpt_path = os.path.join(run_dir, "best.pt")
        torch.save({
            "state_dict": best_state,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "config": {
                "conv_layers": conv_layers,
                "fc_layers": fc_layers,
                "optimizer": optimizer_name,
                "activation": activation,
                "seed": seed,
            }
        }, ckpt_path)


    best_row = min(history, key=lambda r: r["val_loss"])
    out = {
        "conv_layers": conv_layers,
        "fc_layers": fc_layers,
        "optimizer": optimizer_name,
        "activation": activation,
        "seed": seed,
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_row["val_loss"]),
        "best_val_acc": float(best_row["val_acc"]),
        "best_val_bal_acc": float(best_row["val_bal_acc"]),
        "best_val_f1": float(best_row["val_f1"]),
        "best_val_auc": float(best_row["val_auc"]),
        "run_dir": run_dir,
    }
    return out


In [ ]:
EXP_ROOT = "/content/exp_cnn81"
os.makedirs(EXP_ROOT, exist_ok=True)

all_rows = []
best_overall = None

configs = list(product(CONV_LAYERS, FC_LAYERS, OPTIMIZERS, ACTIVATIONS))
print("Total configs:", len(configs))

for (c, f, opt_name, act_name) in configs:
    for seed in SEEDS:
        cfg_id = f"C{c}_F{f}_O{opt_name}_A{act_name}_S{seed}"
        run_dir = os.path.join(EXP_ROOT, cfg_id)

        print("\nRunning:", cfg_id)
        row = train_one_run(c, f, opt_name, act_name, seed, run_dir)
        all_rows.append(row)


        if best_overall is None or row["best_val_loss"] < best_overall["best_val_loss"]:
            best_overall = row


raw_df = pd.DataFrame(all_rows)
raw_path = os.path.join(EXP_ROOT, "results_raw.csv")
raw_df.to_csv(raw_path, index=False)
print("Saved:", raw_path)

print("\nBest overall (by val_loss):")
print(best_overall)


Total configs: 81

Running: C1_F1_Oadam_Arelu_S0

Running: C1_F1_Oadam_Arelu_S1

Running: C1_F1_Oadam_Arelu_S2

Running: C1_F1_Oadam_Aleakyrelu_S0

Running: C1_F1_Oadam_Aleakyrelu_S1

Running: C1_F1_Oadam_Aleakyrelu_S2

Running: C1_F1_Oadam_Asigmoid_S0

Running: C1_F1_Oadam_Asigmoid_S1

Running: C1_F1_Oadam_Asigmoid_S2

Running: C1_F1_Osgd_Arelu_S0

Running: C1_F1_Osgd_Arelu_S1

Running: C1_F1_Osgd_Arelu_S2

Running: C1_F1_Osgd_Aleakyrelu_S0

Running: C1_F1_Osgd_Aleakyrelu_S1

Running: C1_F1_Osgd_Aleakyrelu_S2

Running: C1_F1_Osgd_Asigmoid_S0

Running: C1_F1_Osgd_Asigmoid_S1

Running: C1_F1_Osgd_Asigmoid_S2

Running: C1_F1_Ormsprop_Arelu_S0

Running: C1_F1_Ormsprop_Arelu_S1

Running: C1_F1_Ormsprop_Arelu_S2

Running: C1_F1_Ormsprop_Aleakyrelu_S0

Running: C1_F1_Ormsprop_Aleakyrelu_S1

Running: C1_F1_Ormsprop_Aleakyrelu_S2

Running: C1_F1_Ormsprop_Asigmoid_S0

Running: C1_F1_Ormsprop_Asigmoid_S1

Running: C1_F1_Ormsprop_Asigmoid_S2

Running: C1_F2_Oadam_Arelu_S0

Running: C1_F2_Oadam_Ar

In [ ]:
raw_df = pd.read_csv(os.path.join(EXP_ROOT, "results_raw.csv"))

group_cols = ["conv_layers", "fc_layers", "optimizer", "activation"]
metric_cols = ["best_val_loss", "best_val_acc", "best_val_bal_acc", "best_val_f1", "best_val_auc"]

agg = raw_df.groupby(group_cols)[metric_cols].agg(["mean", "std"]).reset_index()


agg.columns = [
    "_".join(col).strip("_") if isinstance(col, tuple) else col
    for col in agg.columns
]

agg_path = os.path.join(EXP_ROOT, "results_agg.csv")
agg.to_csv(agg_path, index=False)
print("Saved:", agg_path)


display(agg.sort_values("best_val_loss_mean").head(10))


Saved: /content/exp_cnn81/results_agg.csv


,conv_layers,fc_layers,optimizer,activation,best_val_loss_mean,best_val_loss_std,best_val_acc_mean,best_val_acc_std,best_val_bal_acc_mean,best_val_bal_acc_std,best_val_f1_mean,best_val_f1_std,best_val_auc_mean,best_val_auc_std
45,2,3,adam,leakyrelu,0.471068,0.033819,0.807292,0.009021,0.791624,0.022775,0.844582,0.013807,0.840684,0.037098
28,2,1,adam,relu,0.478678,0.041752,0.802083,0.018042,0.796923,0.041901,0.834835,0.002601,0.841368,0.033303
27,2,1,adam,leakyrelu,0.479244,0.044295,0.802083,0.039322,0.794530,0.052188,0.835991,0.031030,0.838632,0.027316
73,3,3,adam,relu,0.485904,0.025457,0.781250,0.031250,0.760684,0.018126,0.824753,0.037425,0.828034,0.033929
36,2,2,adam,leakyrelu,0.486890,0.040417,0.765625,0.068108,0.721538,0.099134,0.829192,0.034823,0.843077,0.020487
63,3,2,adam,leakyrelu,0.488140,0.030061,0.770833,0.023868,0.744957,0.026806,0.821008,0.019371,0.832821,0.027654
55,3,1,adam,relu,0.488313,0.053446,0.776042,0.023868,0.761197,0.027551,0.818652,0.017795,0.836239,0.036121
18,1,3,adam,leakyrelu,0.491017,0.055421,0.776042,0.070457,0.770769,0.075861,0.812486,0.057658,0.829744,0.034279
46,2,3,adam,relu,0.491111,0.060749,0.786458,0.047735,0.786496,0.043105,0.816978,0.045785,0.835897,0.042832
54,3,1,adam,leakyrelu,0.491820,0.042525,0.786458,0.023868,0.781709,0.019584,0.820648,0.023457,0.834188,0.024623


from matplotlib import pyplot as plt
_df_0['index'].plot(kind='hist', bins=20, title='index')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['conv_layers'].plot(kind='hist', bins=20, title='conv_layers')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2['fc_layers'].plot(kind='hist', bins=20, title='fc_layers')
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_18['activation'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_18, x='conv_layers', y='activation', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_19['activation'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_19, x='fc_layers', y='activation', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_20['activation'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_20, x='best_val_loss_mean', y='activation', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:

raw_df = pd.read_csv(os.path.join(EXP_ROOT, "results_raw.csv"))
best_row = raw_df.sort_values("best_val_loss").iloc[0].to_dict()

print("Best row:", best_row)


src = os.path.join(best_row["run_dir"], "best.pt")
dst = os.path.join(EXP_ROOT, "best_model.pt")
import shutil
shutil.copy(src, dst)
print("Saved best model to:", dst)


Best row: {'conv_layers': 1, 'fc_layers': 2, 'optimizer': 'adam', 'activation': 'leakyrelu', 'seed': 2, 'best_epoch': 10, 'best_val_loss': 0.4346218705177307, 'best_val_acc': 0.8125, 'best_val_bal_acc': 0.803076923076923, 'best_val_f1': 0.8461538461538461, 'best_val_auc': 0.8512820512820513, 'run_dir': '/content/exp_cnn81/C1_F2_Oadam_Aleakyrelu_S2'}
Saved best model to: /content/exp_cnn81/best_model.pt


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

raw_path = "/content/exp_cnn81/results_raw.csv"
raw = pd.read_csv(raw_path)

outdir = Path("chap5_outputs")
outdir.mkdir(exist_ok=True)

metric = "best_val_bal_acc"

def interaction_plot(df, x_col, line_col, avg_over_cols, title, out_png):

    per_seed = (
        df.groupby([x_col, line_col, "seed"])[metric]
          .mean()
          .reset_index()
    )


    summary = (
        per_seed.groupby([x_col, line_col])[metric]
                .agg(["mean", "std"])
                .reset_index()
    )

    x_levels = sorted(summary[x_col].unique())
    line_levels = list(summary[line_col].unique())

    plt.figure()
    for lv in line_levels:
        sub = summary[summary[line_col] == lv].sort_values(x_col)
        xs = sub[x_col].values
        ys = sub["mean"].values
        es = sub["std"].values
        plt.errorbar(xs, ys, yerr=es, marker="o", capsize=3, label=str(lv))

    plt.xticks(x_levels)
    plt.xlabel(x_col)
    plt.ylabel("Mean validation balanced accuracy (±1 SD across seeds)")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outdir / out_png, dpi=200)
    plt.close()

# Figure 5.3: C × A
interaction_plot(
    raw, "conv_layers", "activation",
    avg_over_cols=["fc_layers", "optimizer"],
    title="Interaction: Convolutional depth × Activation (C × A)",
    out_png="Figure_5_3_CxA.png"
)

# Figure 5.4: C × O
interaction_plot(
    raw, "conv_layers", "optimizer",
    avg_over_cols=["fc_layers", "activation"],
    title="Interaction: Convolutional depth × Optimiser (C × O)",
    out_png="Figure_5_4_CxO.png"
)

# Figure 5.5: F × O
interaction_plot(
    raw, "fc_layers", "optimizer",
    avg_over_cols=["conv_layers", "activation"],
    title="Interaction: Fully connected depth × Optimiser (F × O)",
    out_png="Figure_5_5_FxO.png"
)

print("Saved figures in:", outdir.resolve())


Saved figures in: /content/chap5_outputs
